# GeoWatch Copilot

### Training notebook -- 6-band (RGB + NIR/SWIR1/SWIR2) dual-stem ResNet50 + DeepLabV3+

Cleaned up in this pass:
- Removed dead/legacy single-split training cells (superseded by the proper LOCO + production cells)
- Removed one-off debugging/plotting cells that assumed 3-band uint8 images (now incompatible with the 6-band float32 tile format)
- Wired in the 6-band NIR/SWIR pipeline end-to-end (Drive sync -> patch builders -> model -> band-stats)


## 0. Install dependencies

In [ ]:
!pip install -q transformers timm einops geopandas shapely scipy pillow matplotlib seaborn scikit-learn scikit-image
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install -q huggingface_hub

print('Dependencies installed.')
# NOTE: scikit-image added this pass -- required by the 6-band patch
# builders (cell 12), which use skimage.transform.resize since PIL
# cannot resize 6-channel float32 arrays.


## 1. Load data from Drive

One cell. Mounts Drive, copies annotations/tiles/result.json and the OSM roads/waterways
geojson (from each run's `osm/` subfolder) into `/content/data/{city}/`, then auto-discovers
which cities are actually usable and pulls their AOI bounding boxes from `result.json`.

**UPDATED THIS PASS:** now also pulls `tile_0_0.npy` -- the 6-band float32 reflectance tile
produced by the updated `tiler.py`'s `generate_tiles()` -- alongside the existing RGB PNG.
This is a hard requirement: without it, every patch-builder function in cell 12 raises
`FileNotFoundError` for that city and silently contributes 0 patches.

Update `CITY_MAP` below if your run IDs change.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil, json

DRIVE_BASE = '/content/drive/MyDrive/Geowatch'
DATA_ROOT = '/content/data'

CITY_MAP = {
    'dharavi': 'dharavi_20260702_163012',
    'nairobi': 'nairobi_20260702_164731',
    'jakarta': 'jakarta_20260702_163609',
    'hcmc':    'hcmc_20260702_163714',
    'kigali':  'kigali_20260702_163814',
    'accra':   'accra_20260702_163854',
    'dhaka':     'dhaka_20260702_163939',
    'lagos':     'lagos_20260702_165430',
    'capetown':  'capetown_20260702_164022',
    'guatemala': 'guatemala_20260702_164206',
    'nusantara': 'nusantara_20260702_165811',
    }

os.makedirs(DATA_ROOT, exist_ok=True)

# tile_0_0.png location differs between pipeline versions:
#   - original 5 cities: run_dir/tile_0_0.png
#   - new 6 cities:       run_dir/tiles/tile_0_0.png
TILE_SUBPATHS = ['tile_0_0.png', 'tiles/tile_0_0.png']
# NEW: 6-band float32 reflectance tile (.npy), produced by tiler.py's
# generate_tiles(). Required by every patch builder in cell 12 -- the
# PNG above is kept only as an RGB preview / legacy fallback, NOT used
# as a training source anymore.
NPY_TILE_SUBPATHS = ['tile_0_0.npy', 'tiles/tile_0_0.npy']

MAIN_FILES_NO_TILE = ['annotations.json', 'result.json']
OSM_FILES = ['roads.geojson', 'waterways.geojson']

missing_npy_cities = []

for city, run_id in CITY_MAP.items():
    run_dir = f'{DRIVE_BASE}/{run_id}'
    dst_dir = f'{DATA_ROOT}/{city}'
    os.makedirs(dst_dir, exist_ok=True)

    if not os.path.isdir(run_dir):
        print(f'X {city}: run dir not found at {run_dir} -- skipping')
        continue

    copied, missing = [], []

    for f in MAIN_FILES_NO_TILE:
        src_path = f'{run_dir}/{f}'
        if os.path.isfile(src_path):
            shutil.copy(src_path, f'{dst_dir}/{f}')
            copied.append(f)
        else:
            missing.append(f)

    tile_found = False
    for sub in TILE_SUBPATHS:
        src_path = f'{run_dir}/{sub}'
        if os.path.isfile(src_path):
            shutil.copy(src_path, f'{dst_dir}/tile_0_0.png')
            copied.append(f'tile_0_0.png (from {sub})')
            tile_found = True
            break
    if not tile_found:
        missing.append('tile_0_0.png (checked: ' + ', '.join(TILE_SUBPATHS) + ')')

    npy_found = False
    for sub in NPY_TILE_SUBPATHS:
        src_path = f'{run_dir}/{sub}'
        if os.path.isfile(src_path):
            shutil.copy(src_path, f'{dst_dir}/tile_0_0.npy')
            copied.append(f'tile_0_0.npy (from {sub})')
            npy_found = True
            break
    if not npy_found:
        missing.append('tile_0_0.npy (checked: ' + ', '.join(NPY_TILE_SUBPATHS) + ')')
        missing_npy_cities.append(city)

    osm_dir = f'{run_dir}/osm'
    for f in OSM_FILES:
        src_path = f'{osm_dir}/{f}'
        if os.path.isfile(src_path):
            shutil.copy(src_path, f'{dst_dir}/{f}')
            copied.append(f)
        else:
            missing.append(f'osm/{f}')

    status = 'OK' if not missing else f'MISSING: {missing}'
    print(f'{city:<10} <- {run_id:<26} [{status}]')

print()
if missing_npy_cities:
    print(f'WARNING: {len(missing_npy_cities)} cities missing tile_0_0.npy '
          f'(6-band tile): {missing_npy_cities}')
    print('  -> these cities will contribute 0 patches until the .npy tile is')
    print('     generated (updated tiler.py generate_tiles()) and uploaded to Drive.')
else:
    print('All 11 cities have tile_0_0.npy (6-band tile).')


In [ ]:
missing_road_osm = []
missing_water_osm = []

for city, run_id in CITY_MAP.items():
    road_src = f'{DRIVE_BASE}/{run_id}/osm_generated_annotations.json'
    road_dst = f'{DATA_ROOT}/{city}/osm_generated_annotations.json'
    if os.path.isfile(road_src):
        shutil.copy(road_src, road_dst)
        road_status = 'OK'
    else:
        missing_road_osm.append(city)
        road_status = 'MISSING'

    water_src = f'{DRIVE_BASE}/{run_id}/osm_generated_annotations_water.json'
    water_dst = f'{DATA_ROOT}/{city}/osm_generated_annotations_water.json'
    if os.path.isfile(water_src):
        shutil.copy(water_src, water_dst)
        water_status = 'OK'
    else:
        missing_water_osm.append(city)
        water_status = 'MISSING'

    print(f'{city:<10} <- {run_id:<26} '
          f'[road:{road_status:<7}] [water:{water_status:<7}]')

print()
if missing_road_osm:
    print(f'{len(missing_road_osm)} cities missing osm_generated_annotations.json (road): {missing_road_osm}')
else:
    print('All 11 cities have osm_generated_annotations.json (road).')

if missing_water_osm:
    print(f'{len(missing_water_osm)} cities missing osm_generated_annotations_water.json (water): {missing_water_osm}')
else:
    print('All 11 cities have osm_generated_annotations_water.json (water).')


In [ ]:
import torch
import numpy as np
import random
import os

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Seed set to {SEED} -- cudnn set to deterministic mode.")


## 1b. Verify and auto-discover

In [ ]:
import os, json

DATA_ROOT = '/content/data'

LOCO_CLEAN_CITIES = [
    'dharavi', 'accra', 'nairobi', 'jakarta', 'hcmc', 'kigali',
    'dhaka', 'lagos', 'capetown', 'guatemala', 'nusantara',
]

CITIES = [
    city for city in LOCO_CLEAN_CITIES
    if os.path.isfile(f'{DATA_ROOT}/{city}/annotations.json')
    and os.path.isfile(f'{DATA_ROOT}/{city}/tile_0_0.png')
    and os.path.isfile(f'{DATA_ROOT}/{city}/tile_0_0.npy')
]

missing = set(LOCO_CLEAN_CITIES) - set(CITIES)
if missing:
    print(f"WARNING: expected 11 clean cities but these are missing files: {missing}")
    print(f"  Check CITY_MAP in cell 4 points to the new run_ids, Drive sync ran, and")
    print(f"  tile_0_0.npy specifically was uploaded (see cell 4's missing_npy_cities warning).")

print(f"Found {len(CITIES)}/11 clean cities for LOCO: {CITIES}")

CITY_BBOXES = {}
for city in CITIES:
    result_path = f'{DATA_ROOT}/{city}/result.json'
    if os.path.isfile(result_path):
        with open(result_path) as f:
            aoi = json.load(f)['aoi']
        CITY_BBOXES[city] = [aoi['west'], aoi['south'], aoi['east'], aoi['north']]
        print(f"  {city}: bbox loaded")
    else:
        print(f"  {city}: no result.json -- OSM patches will be skipped")

print(f"\nReady. {len(CITY_BBOXES)} cities with bboxes.")

if len(CITIES) < 11:
    print("\nStopping point: fix the missing cities above before proceeding to LOCO.")


## 2. Category definitions

In [ ]:
CATEGORIES = [
    'dense_informal_roofing',   # 0
    'sparse_informal_roofing',  # 1
    'paved_road',                # 2
    'standing_water',            # 3
    'vegetation_clearing',       # 4
    'active_construction',       # 5
    'dense_vegetation',          # 6
]
CAT2IDX = {c: i for i, c in enumerate(CATEGORIES)}
NUM_CLASSES = len(CATEGORIES)

IGNORE_INDEX = 255

print(f'{NUM_CLASSES} categories defined: {CATEGORIES}')


## 3. Dataset construction -- five sources

- **SAM**: human-annotated segment crops (real per-pixel masks, all 7 ML categories)
- **OSM (live-rasterized)**: paved roads only (major road types: primary/secondary/
  tertiary/trunk/motorway), rasterized on the fly from each city's roads.geojson
- **OSM-generated (road)**: real, geometrically-correct paved_road masks pre-generated
  from OSM road-vector geometry, covering ALL paved road types including residential/
  unclassified/service roads -- the fix for the paved_road / dense_informal_roofing confusion.
- **OSM-generated (water)**: real, geometrically-correct standing_water masks pre-generated
  from OSM water geometry (waterway centerlines + natural=water/water=* polygons).
- **Sliding window**: dense crops sliced from each city's real per-pixel label canvas.

**UPDATED THIS PASS -- 6-band input (Track A item 2, NIR/SWIR bands):** every source below
now loads the 6-band float32 reflectance `.npy` tile via `load_tile_multiband()` instead of
the 3-band uint8 RGB PNG. Image crops are resized with `skimage.transform.resize` (PIL cannot
resize 6-channel arrays). Mask logic is completely unchanged in every function -- only the
image-loading/cropping side changed.

All five sources write patch dicts with `image` (now shape `(H, W, 6)` float32), `mask`, `city`,
`source`, `label`.


In [ ]:
import json
import numpy as np
from PIL import Image
from pathlib import Path
from skimage.transform import resize as sk_resize
import geopandas as gpd
from shapely.geometry import MultiLineString, LineString
import random

random.seed(42)
np.random.seed(42)

NUM_BANDS = 6  # Blue, Green, Red, NIR, SWIR1, SWIR2 -- must match tiler.py's BAND_NAMES order


def decode_mask_rle(rle: dict) -> np.ndarray:
    """Matches ingestion/segmentation.py's encoder -- keep in sync. UNCHANGED."""
    h, w = rle["size"]
    counts = rle["counts"]
    flat = np.zeros(h * w, dtype=bool)
    idx = 0
    val = False
    for c in counts:
        if val:
            flat[idx:idx + c] = True
        idx += c
        val = not val
    return flat.reshape(w, h).T


def load_tile_multiband(city: str) -> np.ndarray:
    """
    Loads the 6-band training tile as float32 reflectance, shape (H, W, 6).

    Replaces the old `Image.open(f'/content/data/{city}/tile_0_0.png')`
    pattern used everywhere in this notebook. Expects the .npy tile
    produced by the updated tiler.py's generate_tiles() (NOT
    generate_rgb_preview_tiles(), which is PNG/8-bit/RGB-only and must
    not be used as a training source).

    Path convention: /content/data/{city}/tile_0_0.npy
    """
    npy_path = Path(f'/content/data/{city}/tile_0_0.npy')
    if not npy_path.exists():
        raise FileNotFoundError(
            f"[{city}] Expected multi-band tile at {npy_path} -- not found. "
            f"Check that Drive sync pulled the .npy tile (from the updated "
            f"tiler.py's generate_tiles()), not just the legacy PNG."
        )
    arr = np.load(npy_path)  # (H, W, 6) float32
    if arr.shape[-1] != NUM_BANDS:
        raise ValueError(
            f"[{city}] tile_0_0.npy has {arr.shape[-1]} bands, expected {NUM_BANDS}. "
            f"Check tiler.py's BAND_NAMES / S2_BANDS match what was exported."
        )
    return arr


def build_sam_patches(city: str, patch_size: int = 64) -> list:
    """
    UPDATED: loads the real 6-band tile instead of RGB PNG. Crop is now
    (patch_size, patch_size, 6) float32 reflectance instead of
    (patch_size, patch_size, 3) uint8. Mask logic is completely
    unchanged -- only the image side changed.
    """
    ann_path = Path(f'/content/data/{city}/annotations.json')

    if not ann_path.exists():
        print(f'  [{city}] missing annotations.json, skipping.')
        return []

    try:
        tile_arr = load_tile_multiband(city)  # (H, W, 6) float32
    except FileNotFoundError as e:
        print(f'  [{city}] {e}')
        return []

    tile_h, tile_w = tile_arr.shape[:2]

    with open(ann_path) as f:
        ann_data = json.load(f)

    patches_out = []
    skipped, unknown, excluded, no_mask = 0, 0, 0, 0

    for ann in ann_data['annotations']:
        if ann.get('skipped', False) or ann.get('human_label') is None:
            skipped += 1
            continue
        if ann['human_label'] == 'unknown':
            unknown += 1
            continue
        if ann['human_label'] not in CAT2IDX:
            excluded += 1
            continue

        x, y, w, h = ann['bbox']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(tile_w, x + w), min(tile_h, y + h)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = tile_arr[y1:y2, x1:x2, :]  # (h, w, 6) float32
        crop_arr = sk_resize(
            crop, (patch_size, patch_size), order=1,
            preserve_range=True, anti_aliasing=True, channel_axis=-1,
        ).astype(np.float32)

        label_idx = CAT2IDX[ann['human_label']]

        mask_rle = ann.get('mask_rle')
        if mask_rle is not None:
            full_mask = decode_mask_rle(mask_rle)
            local_mask = full_mask[y1:y2, x1:x2]
            local_mask_img = Image.fromarray(local_mask.astype(np.uint8) * 255)
            local_mask_resized = local_mask_img.resize((patch_size, patch_size), Image.NEAREST)
            local_mask_arr = np.array(local_mask_resized) > 127

            label_mask = np.full((patch_size, patch_size), IGNORE_INDEX, dtype=np.uint8)
            label_mask[local_mask_arr] = label_idx
        else:
            no_mask += 1
            label_mask = np.full((patch_size, patch_size), label_idx, dtype=np.uint8)

        patches_out.append({
            'image': crop_arr,       # (64, 64, 6) float32 reflectance
            'mask': label_mask,
            'city': city,
            'source': 'sam',
            'label': ann['human_label'],
        })

    print(f'  [{city}] SAM patches: {len(patches_out)} usable | {skipped} skipped | '
          f'{unknown} unknown excluded | {excluded} OSM-only-category excluded | '
          f'{no_mask} used bbox fallback (no mask_rle)')
    return patches_out


def build_osm_patches(city: str, patch_size: int = 64, road_buffer_px: int = 2) -> list:
    """
    Source 2: OSM-derived training masks -- PAVED ROADS ONLY.

    UPDATED: tile_arr now comes from load_tile_multiband() -- (H, W, 6)
    float32 -- instead of a PIL-loaded uint8 RGB array. The raw numpy
    slicing is unchanged and works identically on the 6-band array.

    This version labels ONLY pixels within `road_buffer_px` of the
    traced road line as paved_road. Everything else in the crop is set
    to IGNORE_INDEX (excluded from loss) rather than guessed at.

    NOTE: this source only covers PAVED_TYPES = primary/secondary/
    tertiary/trunk/motorway (+ links) -- it deliberately does NOT
    include residential/unclassified/service roads. build_osm_generated_
    patches (below) covers those, using real buffered-geometry masks
    instead of thin traced lines.
    """
    roads_path = Path(f'/content/data/{city}/roads.geojson')

    try:
        tile_arr = load_tile_multiband(city)  # (H, W, 6) float32
    except FileNotFoundError as e:
        print(f'  [{city}] {e}')
        return []

    tile_h, tile_w = tile_arr.shape[:2]

    west, south, east, north = CITY_BBOXES[city]
    lon_per_px = (east - west) / tile_w
    lat_per_px = (north - south) / tile_h

    aoi_km2 = (east - west) * 111 * (north - south) * 111
    sample_stride = max(20, int(aoi_km2 * 2))
    print(f'  [{city}] AOI ~{aoi_km2:.1f} km^2 -> road_stride={sample_stride}')

    def lon_to_px(lon):
        return (lon - west) / lon_per_px

    def lat_to_py(lat):
        return (north - lat) / lat_per_px

    PAVED_TYPES = {'primary', 'secondary', 'tertiary', 'trunk', 'motorway',
                   'primary_link', 'secondary_link', 'tertiary_link'}

    patches_out = []
    half = patch_size // 2

    def rasterize_lines(gdf, label_idx, label_name, stride):
        road_pixels = []
        all_road_pixels_global = []
        for _, row in gdf.iterrows():
            geom = row.geometry
            if geom is None:
                continue
            if isinstance(geom, MultiLineString):
                line_list = list(geom.geoms)
            elif isinstance(geom, LineString):
                line_list = [geom]
            else:
                continue

            for line in line_list:
                try:
                    coords = list(line.coords)
                except Exception:
                    continue
                for i in range(len(coords) - 1):
                    x0 = lon_to_px(coords[i][0])
                    y0 = lat_to_py(coords[i][1])
                    x1 = lon_to_px(coords[i+1][0])
                    y1 = lat_to_py(coords[i+1][1])
                    n = max(int(np.hypot(x1-x0, y1-y0)) * 2, 2)
                    xs = np.linspace(x0, x1, n)
                    ys = np.linspace(y0, y1, n)
                    for x, y in zip(xs, ys):
                        xi, yi = int(round(x)), int(round(y))
                        all_road_pixels_global.append((xi, yi))
                        if half <= xi < tile_w - half and half <= yi < tile_h - half:
                            road_pixels.append((xi, yi))

        if not road_pixels:
            return []

        road_pixel_set = set(all_road_pixels_global)

        sampled = road_pixels[::stride]
        result = []
        for cx, cy in sampled:
            crop_arr = tile_arr[cy-half:cy+half, cx-half:cx+half, :]  # (patch_size, patch_size, 6)
            if crop_arr.shape[:2] != (patch_size, patch_size):
                continue

            label_mask = np.full((patch_size, patch_size), IGNORE_INDEX, dtype=np.uint8)
            for rx, ry in road_pixel_set:
                if (cx - half - road_buffer_px) <= rx <= (cx + half + road_buffer_px) and \
                   (cy - half - road_buffer_px) <= ry <= (cy + half + road_buffer_px):
                    local_x = rx - (cx - half)
                    local_y = ry - (cy - half)
                    x_lo = max(0, local_x - road_buffer_px)
                    x_hi = min(patch_size, local_x + road_buffer_px + 1)
                    y_lo = max(0, local_y - road_buffer_px)
                    y_hi = min(patch_size, local_y + road_buffer_px + 1)
                    if x_hi > x_lo and y_hi > y_lo:
                        label_mask[y_lo:y_hi, x_lo:x_hi] = label_idx

            if not (label_mask == label_idx).any():
                continue

            result.append({
                'image': crop_arr.astype(np.float32),
                'mask': label_mask,
                'city': city,
                'source': 'osm',
                'label': label_name,
            })
        return result

    if roads_path.exists():
        try:
            roads_gdf = gpd.read_file(roads_path)
            hw_col = 'highway' if 'highway' in roads_gdf.columns else None
            if hw_col:
                paved = roads_gdf[roads_gdf[hw_col].isin(PAVED_TYPES)]
                if not paved.empty:
                    p = rasterize_lines(paved, CAT2IDX['paved_road'], 'paved_road', sample_stride)
                    PAVED_ROAD_CAP_PER_CITY = 25
                    if len(p) > PAVED_ROAD_CAP_PER_CITY:
                        p = random.sample(p, PAVED_ROAD_CAP_PER_CITY)
                    patches_out.extend(p)
                    print(f'  [{city}] OSM paved road patches: {len(p)} '
                          f'(thin-line masks, buffer={road_buffer_px}px, capped at {PAVED_ROAD_CAP_PER_CITY})')
            else:
                print(f'  [{city}] roads.geojson has no highway column -- columns: {list(roads_gdf.columns)}')
        except Exception as e:
            print(f'  [{city}] roads.geojson error: {e}')
    else:
        print(f'  [{city}] roads.geojson not found')

    if not patches_out:
        print(f'  [{city}] No OSM patches generated')

    return patches_out


def build_osm_generated_patches(city: str, patch_size: int = 64,
                                 max_per_city: int = 25) -> list:
    """
    Source 4: real, geometrically-correct paved_road masks generated
    directly from OSM road-vector geometry. Same schema as
    annotations.json (mask_rle, human_label, bbox), mirrors
    build_sam_patches's logic almost exactly.

    UPDATED: image loading now goes through load_tile_multiband() +
    skimage resize instead of PIL, matching build_sam_patches.

    CAPPED AT 25/city -- deliberately kept at the same scale as
    build_osm_patches's existing cap, not larger, to avoid recreating
    the exact "one source dominates the dataset" imbalance problem this
    project already fixed once before. Sampling is spatially spread
    (greedy farthest-point selection by segment centroid).
    """
    ann_path = Path(f'/content/data/{city}/osm_generated_annotations.json')

    if not ann_path.exists():
        print(f'  [{city}] no osm_generated_annotations.json, skipping.')
        return []

    try:
        tile_arr = load_tile_multiband(city)  # (H, W, 6) float32
    except FileNotFoundError as e:
        print(f'  [{city}] {e}')
        return []

    tile_h, tile_w = tile_arr.shape[:2]

    with open(ann_path) as f:
        ann_data = json.load(f)

    anns = [a for a in ann_data['annotations']
            if not a.get('skipped', False)
            and a.get('human_label') in CAT2IDX]

    if not anns:
        print(f'  [{city}] no usable OSM-generated segments.')
        return []

    def centroid(ann):
        x, y, w, h = ann['bbox']
        return (x + w / 2, y + h / 2)

    if len(anns) > max_per_city:
        random.shuffle(anns)
        selected = [anns.pop()]
        while len(selected) < max_per_city and anns:
            sel_centroids = [centroid(a) for a in selected]
            best_idx, best_dist = 0, -1
            for i, a in enumerate(anns):
                cx, cy = centroid(a)
                min_dist = min(
                    (cx - sx) ** 2 + (cy - sy) ** 2 for sx, sy in sel_centroids
                )
                if min_dist > best_dist:
                    best_dist, best_idx = min_dist, i
            selected.append(anns.pop(best_idx))
        anns = selected

    patches_out = []
    for ann in anns:
        x, y, w, h = ann['bbox']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(tile_w, x + w), min(tile_h, y + h)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = tile_arr[y1:y2, x1:x2, :]
        crop_arr = sk_resize(
            crop, (patch_size, patch_size), order=1,
            preserve_range=True, anti_aliasing=True, channel_axis=-1,
        ).astype(np.float32)

        label_idx = CAT2IDX[ann['human_label']]

        full_mask = decode_mask_rle(ann['mask_rle'])
        local_mask = full_mask[y1:y2, x1:x2]
        local_mask_img = Image.fromarray(local_mask.astype(np.uint8) * 255)
        local_mask_resized = local_mask_img.resize((patch_size, patch_size), Image.NEAREST)
        local_mask_arr = np.array(local_mask_resized) > 127

        label_mask = np.full((patch_size, patch_size), IGNORE_INDEX, dtype=np.uint8)
        label_mask[local_mask_arr] = label_idx

        patches_out.append({
            'image': crop_arr,
            'mask': label_mask,
            'city': city,
            'source': 'osm_generated',
            'label': ann['human_label'],
        })

    print(f'  [{city}] OSM-generated road patches: {len(patches_out)} '
          f'(spatially-spread sample, capped at {max_per_city})')
    return patches_out


def build_osm_generated_water_patches(city: str, patch_size: int = 64,
                                       max_per_city: int = 25) -> list:
    """
    Source 5: real, geometrically-correct standing_water masks generated
    directly from OSM water geometry -- direct mirror of
    build_osm_generated_patches (Source 4, paved_road) above, just
    pointed at osm_generated_annotations_water.json and standing_water
    instead.

    UPDATED: same image-loading swap as build_osm_generated_patches --
    load_tile_multiband() + skimage resize instead of PIL. Kigali's
    known 0-segment gap (all connected components fell below the 15px
    noise-filter in merge_osm_water_masks.py) is unaffected by this
    change -- it will still correctly return [] for Kigali since that's
    a data-availability issue, not a loading issue.
    """
    ann_path = Path(f'/content/data/{city}/osm_generated_annotations_water.json')

    if not ann_path.exists():
        print(f'  [{city}] no osm_generated_annotations_water.json, skipping.')
        return []

    try:
        tile_arr = load_tile_multiband(city)  # (H, W, 6) float32
    except FileNotFoundError as e:
        print(f'  [{city}] {e}')
        return []

    tile_h, tile_w = tile_arr.shape[:2]

    with open(ann_path) as f:
        ann_data = json.load(f)

    anns = [a for a in ann_data['annotations']
            if not a.get('skipped', False)
            and a.get('human_label') in CAT2IDX]

    if not anns:
        print(f'  [{city}] no usable OSM-generated water segments.')
        return []

    def centroid(ann):
        x, y, w, h = ann['bbox']
        return (x + w / 2, y + h / 2)

    if len(anns) > max_per_city:
        random.shuffle(anns)
        selected = [anns.pop()]
        while len(selected) < max_per_city and anns:
            sel_centroids = [centroid(a) for a in selected]
            best_idx, best_dist = 0, -1
            for i, a in enumerate(anns):
                cx, cy = centroid(a)
                min_dist = min(
                    (cx - sx) ** 2 + (cy - sy) ** 2 for sx, sy in sel_centroids
                )
                if min_dist > best_dist:
                    best_dist, best_idx = min_dist, i
            selected.append(anns.pop(best_idx))
        anns = selected

    patches_out = []
    for ann in anns:
        x, y, w, h = ann['bbox']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(tile_w, x + w), min(tile_h, y + h)
        if x2 <= x1 or y2 <= y1:
            continue

        crop = tile_arr[y1:y2, x1:x2, :]
        crop_arr = sk_resize(
            crop, (patch_size, patch_size), order=1,
            preserve_range=True, anti_aliasing=True, channel_axis=-1,
        ).astype(np.float32)

        label_idx = CAT2IDX[ann['human_label']]

        full_mask = decode_mask_rle(ann['mask_rle'])
        local_mask = full_mask[y1:y2, x1:x2]
        local_mask_img = Image.fromarray(local_mask.astype(np.uint8) * 255)
        local_mask_resized = local_mask_img.resize((patch_size, patch_size), Image.NEAREST)
        local_mask_arr = np.array(local_mask_resized) > 127

        label_mask = np.full((patch_size, patch_size), IGNORE_INDEX, dtype=np.uint8)
        label_mask[local_mask_arr] = label_idx

        patches_out.append({
            'image': crop_arr,
            'mask': label_mask,
            'city': city,
            'source': 'osm_generated_water',
            'label': ann['human_label'],
        })

    print(f'  [{city}] OSM-generated water patches: {len(patches_out)} '
          f'(spatially-spread sample, capped at {max_per_city})')
    return patches_out


def build_tile_label_canvas(city: str) -> tuple:
    """
    Paints every annotated segment's REAL per-pixel mask onto a single
    full-tile-resolution label canvas, once per city.

    UPDATED: this function doesn't touch the image at all (only the
    label canvas), so the only change needed is getting tile_w/tile_h
    from load_tile_multiband(city).shape[:2] instead of
    Image.open(...).size. Everything else -- segment sorting,
    largest-area-first painting, bbox fallback -- is UNCHANGED.
    """
    ann_path = Path(f'/content/data/{city}/annotations.json')

    if not ann_path.exists():
        return None, 0, 0

    try:
        tile_h, tile_w = load_tile_multiband(city).shape[:2]
    except FileNotFoundError:
        return None, 0, 0

    with open(ann_path) as f:
        ann_data = json.load(f)

    canvas = np.full((tile_h, tile_w), IGNORE_INDEX, dtype=np.uint8)

    usable = []
    for ann in ann_data['annotations']:
        if ann.get('skipped', False) or ann.get('human_label') is None:
            continue
        if ann['human_label'] == 'unknown':
            continue
        if ann['human_label'] not in CAT2IDX:
            continue
        usable.append(ann)

    usable.sort(key=lambda a: a.get('area', 0), reverse=True)

    n_real_mask, n_bbox_fallback = 0, 0

    for ann in usable:
        label_idx = CAT2IDX[ann['human_label']]
        x, y, w, h = ann['bbox']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = min(tile_w, x + w), min(tile_h, y + h)
        if x2 <= x1 or y2 <= y1:
            continue

        mask_rle = ann.get('mask_rle')
        if mask_rle is not None:
            full_mask = decode_mask_rle(mask_rle)  # (tile_h, tile_w) bool
            canvas[full_mask] = label_idx
            n_real_mask += 1
        else:
            canvas[y1:y2, x1:x2] = label_idx
            n_bbox_fallback += 1

    print(f'  [{city}] Label canvas built: {n_real_mask} real masks, '
          f'{n_bbox_fallback} bbox fallback, {len(usable)} total segments painted')

    return canvas, tile_w, tile_h


def build_sliding_window_patches(city: str, patch_size: int = 64, stride: int = 32,
                                  min_labeled_pct: float = 5.0,
                                  max_per_class_per_city: int = 30) -> list:
    """
    Source 3: Sliding window patches, sliced from the real per-pixel
    label canvas (see build_tile_label_canvas).

    UPDATED: tile_arr now comes from load_tile_multiband() -- (H, W, 6)
    float32 -- instead of a PIL-loaded uint8 RGB array. The crop slicing
    is raw numpy indexing and needs no other change. No resize is
    needed here (patches are cut directly at patch_size).
    max_per_class_per_city capping logic is UNCHANGED.
    """
    from collections import defaultdict

    canvas, tile_w, tile_h = build_tile_label_canvas(city)
    if canvas is None:
        return []

    try:
        tile_arr = load_tile_multiband(city)  # (H, W, 6) float32
    except FileNotFoundError as e:
        print(f'  [{city}] {e}')
        return []

    candidates_by_label = defaultdict(list)
    min_labeled_px = int(patch_size * patch_size * min_labeled_pct / 100.0)

    for y_start in range(0, tile_h - patch_size, stride):
        for x_start in range(0, tile_w - patch_size, stride):
            x_end = x_start + patch_size
            y_end = y_start + patch_size

            label_mask = canvas[y_start:y_end, x_start:x_end]
            n_labeled = (label_mask != IGNORE_INDEX).sum()
            if n_labeled < min_labeled_px:
                continue

            crop_arr = tile_arr[y_start:y_end, x_start:x_end, :].astype(np.float32)

            vals, counts = np.unique(label_mask[label_mask != IGNORE_INDEX], return_counts=True)
            dominant_idx = int(vals[np.argmax(counts)]) if len(vals) else IGNORE_INDEX
            dominant_label = CATEGORIES[dominant_idx] if dominant_idx != IGNORE_INDEX else 'unknown'

            candidates_by_label[dominant_label].append({
                'image': crop_arr,
                'mask': label_mask.copy(),
                'city': city,
                'source': 'sliding_window',
                'label': dominant_label,
            })

    patches_out = []
    capped_summary = []
    for label, items in candidates_by_label.items():
        if len(items) > max_per_class_per_city:
            kept = random.sample(items, max_per_class_per_city)
            capped_summary.append(f'{label}: {len(items)}->{max_per_class_per_city}')
        else:
            kept = items
        patches_out.extend(kept)

    cap_note = f' | capped: {", ".join(capped_summary)}' if capped_summary else ''
    print(f'  [{city}] Sliding window patches: {len(patches_out)} '
          f'(min {min_labeled_pct}% coverage, max {max_per_class_per_city}/class){cap_note}')
    return patches_out


Build the full dataset from all cities:

In [ ]:
from collections import Counter

all_patches = []

print('Building dataset from all cities...')
for city in CITIES:
    print(f'\n--- {city.upper()} ---')
    all_patches.extend(build_sam_patches(city))
    all_patches.extend(build_osm_patches(city))
    all_patches.extend(build_osm_generated_patches(city))
    all_patches.extend(build_osm_generated_water_patches(city))
    all_patches.extend(build_sliding_window_patches(city))

print(f'\nRaw total: {len(all_patches)} patches')

label_dist = Counter(p['label'] for p in all_patches)
source_dist = Counter(p['source'] for p in all_patches)

print('\nLabel distribution:')
for cat in CATEGORIES:
    count = label_dist.get(cat, 0)
    bar = chr(9608) * min(count // 2, 40)
    print(f'  {cat:<30} {count:>4}  {bar}')

print('\nSource distribution:')
for src, count in source_dist.items():
    print(f'  {src:<20} {count}')

random.shuffle(all_patches)

for label in CATEGORIES:
    if label_dist.get(label, 0) == 0:
        print(f'  WARNING: {label} has 0 patches -- will be ignored in training')

print(f'\nFinal: {len(all_patches)} patches (no oversampling applied)')


## 3b. Compute real per-band normalization stats

**NEW THIS PASS.** Computes real mean/std per band across the actual 6-band training patches
just built above -- required before training, since the model (Section 6) normalizes using
`BAND_MEAN`/`BAND_STD`, which start as placeholder guesses.

**Do not skip this. Do not train with the placeholder values in Section 6's cell.**
Run this cell, copy the printed `BAND_MEAN =` / `BAND_STD =` lines, and paste them into
`GeoWatchDatasetResNet` in the model cell below, replacing the placeholders there.


In [ ]:
import numpy as np

def compute_band_stats(patches: list) -> tuple:
    n_bands = patches[0]['image'].shape[-1]
    px_sum = np.zeros(n_bands, dtype=np.float64)
    px_sumsq = np.zeros(n_bands, dtype=np.float64)
    total_px = 0
    for p in patches:
        img = p['image'].astype(np.float64)
        px_sum += img.sum(axis=(0, 1))
        px_sumsq += (img ** 2).sum(axis=(0, 1))
        total_px += img.shape[0] * img.shape[1]
    mean = px_sum / total_px
    var = (px_sumsq / total_px) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-8))
    return mean, std

band_names = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
mean, std = compute_band_stats(all_patches)
print('Computed per-band stats across all_patches:')
for name, m, s in zip(band_names, mean, std):
    print(f'  {name:8s} mean={m:.4f} std={s:.4f}')

print('\nPaste these into GeoWatchDatasetResNet in the model cell below (Section 6):')
print(f"  BAND_MEAN = torch.tensor([{', '.join(f'{m:.4f}' for m in mean)}]).view(6, 1, 1)")
print(f"  BAND_STD = torch.tensor([{', '.join(f'{s:.4f}' for s in std)}]).view(6, 1, 1)")


## 4. Class weights

Computed from the **actual built dataset** (`all_patches`), not from annotation counts alone.


In [ ]:
from collections import Counter
import numpy as np
import torch

actual_counts = Counter(p['label'] for p in all_patches)
counts = np.array([actual_counts.get(c, 1) for c in CATEGORIES], dtype=np.float32)
counts = np.maximum(counts, 1)
weights = 1.0 / counts
weights = weights / weights.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32)

print('Recomputed class weights from ACTUAL patch counts:')
for cat, w, c in zip(CATEGORIES, CLASS_WEIGHTS, counts):
    print(f'  {cat:<30} count={int(c):>4}  weight={w:.3f}')


## 5. Data augmentation and PyTorch Dataset

Base `GeoWatchDataset` class -- subclassed by `GeoWatchDatasetResNet` (Section 6) which
overrides `__getitem__` for the real 6-band float32 pipeline. This base class is kept only
for the subclass relationship; its own `__getitem__` (3-band, /255, ImageNet stats) is never
actually used directly since every real training cell below (LOCO, production) instantiates
`GeoWatchDatasetResNet`, not this base class.

**REMOVED THIS PASS:** the old single-split (`VAL_CITY='kigali'`) train/val loader setup and
sample-patch visualization that used to live in this cell. That was a leftover from before
proper LOCO methodology existed (Section 8) -- a single hold-out city is exactly the
"single-fold-as-generalization" anti-pattern this project's own standards reject, and it's
superseded by the real LOCO loop and the production training cell below, both of which build
their own loaders per fold/run. Kept unremoved would also have silently broken this pass,
since it plots `p['image']` directly via `imshow`, which doesn't work on 6-channel float arrays.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import random

PATCH_SIZE = 64


class GeoWatchDataset(Dataset):
    """
    Base PyTorch dataset wrapping the multi-source patch list. Subclassed
    by GeoWatchDatasetResNet (Section 6) for the real 6-band pipeline --
    this base __getitem__ is not used directly in this notebook's actual
    training cells, kept only so the subclass relationship holds.
    """

    def __init__(self, patches: list, augment: bool = True):
        self.patches = patches
        self.augment = augment

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        item = self.patches[idx]
        image = torch.from_numpy(item['image']).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(item['mask']).long()
        return image, mask


print('GeoWatchDataset base class defined (subclassed by GeoWatchDatasetResNet in Section 6).')


## 6. ResNet50 (Sentinel-2 RGB pretrained, dual-stem 6-band) + DeepLabV3+-style decoder

**UPDATED THIS PASS (Track A item 2, NIR/SWIR bands):** dual-stem design -- the pretrained
3-channel RGB stem (SSL4EO-S12 MoCo) is kept byte-for-byte untouched, fed only R/G/B; a new
from-scratch stem handles NIR/SWIR1/SWIR2; both stride-4 feature maps are concatenated and
fused via a 1x1 conv before entering the pretrained `layer1`, so the rest of the pretrained
trunk (`layer1`-`layer4`) never sees a channel-count mismatch.

**IMPORTANT:** `GeoWatchDatasetResNet.BAND_MEAN`/`BAND_STD` below are still PLACEHOLDER
values. Run Section 3b's band-stats cell first, then paste the real printed values in here
before training.


In [ ]:
!pip install -q torchgeo


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchgeo.models import ResNet50_Weights, resnet50


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling -- standard DeepLabV3+ component. UNCHANGED."""
    def __init__(self, in_channels: int, out_channels: int = 256, rates=(1, 6, 12, 18)):
        super().__init__()
        self.branches = nn.ModuleList()
        for rate in rates:
            if rate == 1:
                self.branches.append(nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True),
                ))
            else:
                self.branches.append(nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, kernel_size=3,
                              padding=rate, dilation=rate, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True),
                ))
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_channels * (len(rates) + 1), out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )

    def forward(self, x):
        size = x.shape[-2:]
        feats = [branch(x) for branch in self.branches]
        pooled = self.pool(x)
        pooled = F.interpolate(pooled, size=size, mode='bilinear', align_corners=False)
        feats.append(pooled)
        x = torch.cat(feats, dim=1)
        return self.project(x)


class DeepLabDecoder(nn.Module):
    """UNCHANGED from prior version."""
    def __init__(self, low_level_channels: int, high_level_channels: int,
                 num_classes: int, aspp_channels: int = 256, low_level_proj: int = 48):
        super().__init__()
        self.aspp = ASPP(high_level_channels, out_channels=aspp_channels)
        self.low_level_proj = nn.Sequential(
            nn.Conv2d(low_level_channels, low_level_proj, kernel_size=1, bias=False),
            nn.BatchNorm2d(low_level_proj),
            nn.ReLU(inplace=True),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(aspp_channels + low_level_proj, aspp_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(aspp_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Conv2d(aspp_channels, aspp_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(aspp_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
        )
        self.classifier = nn.Conv2d(aspp_channels, num_classes, kernel_size=1)

    def forward(self, low_level_feat, high_level_feat, target_size):
        x = self.aspp(high_level_feat)
        x = F.interpolate(x, size=low_level_feat.shape[-2:], mode='bilinear', align_corners=False)
        low = self.low_level_proj(low_level_feat)
        x = torch.cat([x, low], dim=1)
        x = self.fuse(x)
        x = self.classifier(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return x


class ExtraBandStem(nn.Module):
    """
    Small from-scratch stem for the 3 non-RGB bands (NIR, SWIR1, SWIR2).
    Mirrors ResNet's own stem shape (7x7 stride 2 conv + BN + ReLU + maxpool)
    so its output spatially matches the pretrained RGB stem's output
    (both end up at stride 4, ready to fuse before layer1). Trained
    entirely from scratch -- no pretraining exists for this input.
    """
    def __init__(self, in_channels: int = 3, out_channels: int = 64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        return x


class GeoWatchResNetSeg(nn.Module):
    """
    ResNet50 encoder pretrained on Sentinel-2 RGB (torchgeo SSL4EO-S12
    MoCo weights) + DeepLabV3+-style decoder.

    Track A item 2: accepts 6-band input (Blue, Green, Red, NIR, SWIR1,
    SWIR2). Option A design:
      - RGB path: the ORIGINAL pretrained 3-channel stem (conv1/bn1/
        relu/maxpool from torchgeo's resnet50) is kept byte-for-byte
        untouched, fed only R/G/B. This preserves 100% of the
        SSL4EO-S12 MoCo pretraining benefit for those 3 channels.
      - Extra-band path: a NEW from-scratch stem (ExtraBandStem) with
        the same output shape, fed NIR/SWIR1/SWIR2.
      - Fusion: the two stride-4 feature maps (64 channels each) are
        concatenated (128 channels) and projected back to 64 channels
        via a 1x1 conv + BN + ReLU, BEFORE entering the pretrained
        layer1.

    Input tensor contract: (B, 6, H, W), channel order
    [Blue, Green, Red, NIR, SWIR1, SWIR2] -- matches tiler.py's
    BAND_NAMES order.
    """

    RGB_CHANNELS = [2, 1, 0]      # R, G, B -- note tiler.py stores Blue,Green,Red so index order matters
    EXTRA_CHANNELS = [3, 4, 5]    # NIR, SWIR1, SWIR2

    def __init__(self, num_classes: int = 7, freeze_encoder: bool = False):
        super().__init__()

        self.encoder = resnet50(weights=ResNet50_Weights.SENTINEL2_RGB_MOCO)
        print('Loaded ResNet50 encoder: torchgeo SSL4EO-S12 MoCo, Sentinel-2 RGB pretrained.')

        self.rgb_stem = nn.Sequential(
            self.encoder.conv1,
            self.encoder.bn1,
            self.encoder.relu,
            self.encoder.maxpool,
        )

        self.extra_stem = ExtraBandStem(in_channels=3, out_channels=64)

        self.stem_fuse = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self._features = {}
        self.encoder.layer1.register_forward_hook(self._hook('low'))
        self.encoder.layer3.register_forward_hook(self._hook('high'))

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
            print('Encoder frozen (not recommended for this backbone -- see docstring).')
        else:
            print('Encoder fully trainable (domain-matched pretraining -- use a low LR).')
            print('NOTE: extra_stem + stem_fuse are NEW, from-scratch params -- '
                  'give them a normal/higher LR, not the low encoder LR meant for '
                  'the pretrained RGB path. See optimizer param groups in training cell.')

        self.decoder = DeepLabDecoder(
            low_level_channels=256,
            high_level_channels=1024,
            num_classes=num_classes,
        )

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        print(f'Parameters: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)')

    def _hook(self, name):
        def fn(module, input, output):
            self._features[name] = output
        return fn

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """pixel_values: (B, 6, H, W) -- [Blue, Green, Red, NIR, SWIR1, SWIR2]."""
        target_size = pixel_values.shape[-2:]

        rgb = pixel_values[:, self.RGB_CHANNELS, :, :]      # (B, 3, H, W)
        extra = pixel_values[:, self.EXTRA_CHANNELS, :, :]  # (B, 3, H, W)

        rgb_feat = self.rgb_stem(rgb)       # (B, 64, H/4, W/4)
        extra_feat = self.extra_stem(extra)  # (B, 64, H/4, W/4)

        fused = torch.cat([rgb_feat, extra_feat], dim=1)  # (B, 128, H/4, W/4)
        fused = self.stem_fuse(fused)                      # (B, 64, H/4, W/4)

        x = self.encoder.layer1(fused)
        x = self.encoder.layer2(x)
        x = self.encoder.layer3(x)
        x = self.encoder.layer4(x)

        low = self._features['low']
        high = self._features['high']
        return self.decoder(low, high, target_size)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
model = GeoWatchResNetSeg(num_classes=NUM_CLASSES, freeze_encoder=False).to(device)
print('Model ready. Expects 6-channel input: [Blue, Green, Red, NIR, SWIR1, SWIR2].')


class GeoWatchDatasetResNet(GeoWatchDataset):
    """
    UPDATED for 6-band float32 reflectance input (was 3-band uint8 RGB).

    - item['image'] is now (patch_size, patch_size, 6) float32 reflectance
      (already scaled ~0-1 range -- NOT 0-255, so no /255.0 division here).
    - Per-band normalization stats (BAND_MEAN / BAND_STD) must be computed
      across the actual 11-city dataset before this is usable for real
      training -- see the band-stats cell (Section 3b). Placeholder values
      below MUST be replaced before trusting any training run that uses them.
    - Augmentation (flips/rotation) applies identically across all 6
      channels via torch ops. Color jitter (brightness/contrast) is now
      RGB-channel-only intentionally -- applying photometric jitter tuned
      for 8-bit visual bands to raw NIR/SWIR reflectance is not meaningful.
    """

    # PLACEHOLDER -- replace with real per-band stats from the band-stats
    # cell (Section 3b) BEFORE trusting any training run using this class.
    BAND_MEAN = torch.tensor([0.15, 0.15, 0.15, 0.30, 0.20, 0.15]).view(6, 1, 1)
    BAND_STD = torch.tensor([0.08, 0.08, 0.08, 0.12, 0.10, 0.09]).view(6, 1, 1)

    def __getitem__(self, idx):
        item = self.patches[idx]
        image = torch.from_numpy(item['image']).permute(2, 0, 1).float()  # (6, H, W), already reflectance-scale
        mask = torch.from_numpy(item['mask']).long()

        if self.augment:
            if random.random() > 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask.unsqueeze(0)).squeeze(0)
            if random.random() > 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask.unsqueeze(0)).squeeze(0)
            k = random.randint(0, 3)
            if k > 0:
                image = torch.rot90(image, k, dims=[1, 2])
                mask = torch.rot90(mask, k, dims=[0, 1])
            if random.random() > 0.5:
                brightness = random.uniform(0.8, 1.2)
                contrast = random.uniform(0.8, 1.2)
                rgb = image[[2, 1, 0], :, :]
                rgb = TF.adjust_brightness(rgb, brightness)
                rgb = TF.adjust_contrast(rgb, contrast)
                image = image.clone()
                image[[2, 1, 0], :, :] = rgb.clamp(0, 1)

        image = (image - self.BAND_MEAN) / self.BAND_STD
        return image, mask


print('GeoWatchDatasetResNet defined -- 6-band, uses BAND_MEAN/BAND_STD.')
print('!!! Replace the placeholder BAND_MEAN/BAND_STD with real stats '
      'computed on your actual 11-city .npy tiles (Section 3b) before '
      'trusting results. !!!')


## 7. Loss functions -- CrossEntropy + Dice + paved_road/dense_informal_roofing separation term

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceLoss(nn.Module):
    """Multiclass Dice loss. Excludes IGNORE_INDEX pixels."""

    def __init__(self, num_classes: int, ignore_index: int = 255, smooth: float = 1.0):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs = F.softmax(logits, dim=1)
        B, C, H, W = probs.shape

        valid_mask = (targets != self.ignore_index)
        targets_clamped = targets.clone()
        targets_clamped[~valid_mask] = 0

        targets_one_hot = F.one_hot(targets_clamped, num_classes=C)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()

        mask_expanded = valid_mask.unsqueeze(1).float()
        probs = probs * mask_expanded
        targets_one_hot = targets_one_hot * mask_expanded

        dice_per_class = []
        for c in range(C):
            p = probs[:, c].reshape(-1)
            t = targets_one_hot[:, c].reshape(-1)
            intersection = (p * t).sum()
            dice = (2.0 * intersection + self.smooth) / (p.sum() + t.sum() + self.smooth)
            dice_per_class.append(1.0 - dice)

        return torch.stack(dice_per_class).mean()


class CombinedLoss(nn.Module):
    """
    CrossEntropy + Dice, equal weighting, PLUS a targeted separation term
    between paved_road and dense_informal_roofing -- fixes the confirmed
    confusion pair (72.9% of all CAAT-unknown-mass pixels, pooled across
    Cape Town + Dharavi, were argmax-confused between exactly these two
    classes). Penalizes the model whenever a ground-truth paved_road or
    dense_informal_roofing pixel's own logit isn't at least
    `separation_margin` above the other class's logit at that pixel.
    """

    def __init__(self, class_weights: torch.Tensor, ignore_index: int = 255,
                 dice_weight: float = 0.5,
                 separation_weight: float = 0.5, separation_margin: float = 2.0,
                 class_a_idx: int = None, class_b_idx: int = None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        self.dice = DiceLoss(num_classes=len(class_weights), ignore_index=ignore_index)
        self.dice_weight = dice_weight
        self.ignore_index = ignore_index

        self.separation_weight = separation_weight
        self.separation_margin = separation_margin
        self.class_a_idx = class_a_idx  # paved_road
        self.class_b_idx = class_b_idx  # dense_informal_roofing

    def separation_loss(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.class_a_idx is None or self.class_b_idx is None:
            return torch.tensor(0.0, device=logits.device)

        valid = targets != self.ignore_index
        logit_a = logits[:, self.class_a_idx, :, :]
        logit_b = logits[:, self.class_b_idx, :, :]

        is_a = valid & (targets == self.class_a_idx)
        is_b = valid & (targets == self.class_b_idx)

        loss = torch.tensor(0.0, device=logits.device)
        n_terms = 0

        if is_a.any():
            gap_a = logit_a[is_a] - logit_b[is_a]
            loss = loss + F.relu(self.separation_margin - gap_a).mean()
            n_terms += 1

        if is_b.any():
            gap_b = logit_b[is_b] - logit_a[is_b]
            loss = loss + F.relu(self.separation_margin - gap_b).mean()
            n_terms += 1

        return loss / n_terms if n_terms > 0 else torch.tensor(0.0, device=logits.device)

    def forward(self, logits: torch.Tensor, targets: torch.Tensor):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        sep_loss = self.separation_loss(logits, targets)

        total = ((1 - self.dice_weight) * ce_loss
                 + self.dice_weight * dice_loss
                 + self.separation_weight * sep_loss)
        return total, ce_loss, dice_loss, sep_loss


criterion = CombinedLoss(
    class_weights=CLASS_WEIGHTS.to(device),
    ignore_index=IGNORE_INDEX,
    dice_weight=0.5,
    separation_weight=0.5,
    separation_margin=2.0,
    class_a_idx=CATEGORIES.index('paved_road'),
    class_b_idx=CATEGORIES.index('dense_informal_roofing'),
)
print('Loss function ready: 0.5 * CrossEntropy (weighted) + 0.5 * Dice '
      '+ 0.5 * pair-separation (paved_road vs dense_informal_roofing)')


In [ ]:
import pickle
import os

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Geowatch/datasets'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

dataset_path = f'{DRIVE_DATASET_DIR}/geowatch_dataset_{len(CITIES)}city_6band.pkl'

with open(dataset_path, 'wb') as f:
    pickle.dump({
        'all_patches': all_patches,
        'cities': CITIES,
        'city_bboxes': CITY_BBOXES,
        'categories': CATEGORIES,
        'class_weights': CLASS_WEIGHTS,
        'num_classes': NUM_CLASSES,
        'ignore_index': IGNORE_INDEX,
        'num_bands': 6,
    }, f)

print(f'Dataset saved: {dataset_path}')
print(f'  {len(all_patches)} total patches across {len(CITIES)} cities')
print('  NOTE: this pkl is now much larger than the old 3-band version -- '
      'each patch stores (64, 64, 6) float32 instead of (64, 64, 3) uint8, '
      'roughly an 8x size increase per patch. This is expected.')

import shutil
local_copy = f'/content/geowatch_dataset_{len(CITIES)}city_6band.pkl'
shutil.copy(dataset_path, local_copy)

from google.colab import files
print('\nTriggering download...')
files.download(local_copy)


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
torch.cuda.empty_cache()


In [ ]:
import shutil
import os
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/Geowatch/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)


## 8. LOCO -- Leave-One-City-Out validation

`run_loco_fold()` trains a fresh `GeoWatchResNetSeg` with one city held out, everything else
as train, early-stopping on val mIoU plateau. This is the real generalization test -- run
BEFORE production training, and always report the full per-fold table, never a bare mean.

**REMOVED THIS PASS:** the old "8. Training loop" / "9. Training curves" cells that used to
live before this section. Those were a single fixed hold-out (`VAL_CITY='kigali'`) manual
training loop from before proper LOCO existed -- exactly the "single-fold-as-generalization"
anti-pattern this project's own standards explicitly reject. They are fully superseded by
this LOCO cell and the production cell below, both of which build their own loaders/model/
optimizer per run rather than relying on any state left over from that old cell.

**REPRODUCIBILITY FIX (this pass):** `run_loco_fold()` now reseeds `random`/`np.random`/`torch`
(CPU+CUDA) at the start of every fold, keyed to a fixed base seed + a stable per-city hash offset.
Previously, each fold inherited whatever RNG state the prior fold left behind, so a fold's result
could differ depending on what order the 11 cities happened to run in -- exactly the kind of
confound that made the earlier water-fix LOCO regression (0.3174 -> 0.2871) impossible to
attribute to a real data effect vs. run-to-run noise. Re-running a single fold in isolation will
now always match its result inside the full 11-fold loop.


In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, f1_score

LOCO_BASE_SEED = 42  # fixed base -- combined with a per-city hash offset inside run_loco_fold


def freeze_bn_stats(module):
    """Puts BatchNorm layers into eval mode so running stats don't get
    overwritten by this fold's small batch statistics."""
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()


def compute_miou(logits: torch.Tensor, targets: torch.Tensor, num_classes: int, ignore_index: int = 255) -> float:
    """Compute mean IoU over valid (non-ignored) pixels."""
    preds = logits.argmax(dim=1)
    valid = targets != ignore_index

    iou_per_class = []
    for c in range(num_classes):
        pred_c = (preds == c) & valid
        true_c = (targets == c) & valid
        intersection = (pred_c & true_c).sum().item()
        union = (pred_c | true_c).sum().item()
        if union == 0:
            continue
        iou_per_class.append(intersection / union)

    return float(np.mean(iou_per_class)) if iou_per_class else 0.0


def compute_per_class_iou(logits: torch.Tensor, targets: torch.Tensor, num_classes: int,
                           ignore_index: int = 255) -> dict:
    """Per-class IoU, not just the mean -- needed to specifically confirm
    whether paved_road / dense_informal_roofing improved."""
    preds = logits.argmax(dim=1)
    valid = targets != ignore_index
    ious = {}
    for c in range(num_classes):
        pred_c = (preds == c) & valid
        true_c = (targets == c) & valid
        intersection = (pred_c & true_c).sum().item()
        union = (pred_c | true_c).sum().item()
        ious[c] = (intersection / union) if union > 0 else None
    return ious


def run_loco_fold(val_city: str, all_patches: list, num_epochs: int = 30,
                   patience: int = 8, batch_size: int = 16,
                   lr_encoder: float = 1e-5, lr_decoder: float = 3e-4,
                   verbose: bool = True) -> dict:
    """
    Trains a fresh GeoWatchResNetSeg with `val_city` held out, everything
    else as train. Early-stops on val mIoU plateau (patience epochs with
    no improvement).

    criterion returns 4 values (total, ce, dice, sep) since CombinedLoss
    includes the paved_road / dense_informal_roofing separation term.
    separation_weight=0.25 (lowered from an earlier 0.5 test that
    overcorrected: paved_road IoU 0.6056 but dense_informal_roofing
    collapsed to 0.0733 on a single-fold sanity check).

    Uses GeoWatchDatasetResNet (6-band) -- confirm BAND_MEAN/BAND_STD in
    that class (Section 6) are the REAL computed values (Section 3b),
    not the placeholders, before trusting this fold's result.

    REPRODUCIBILITY FIX: reseeds random/np.random/torch (CPU+CUDA) at
    the START of every fold, keyed off a fixed base seed + a stable
    per-city offset (hash of val_city name, not fold index -- so the
    seed for a given held-out city doesn't depend on what order cities
    happen to be iterated in). Without this, fold N's weight init /
    DataLoader shuffling draws from whatever RNG state folds 1..N-1
    happened to leave behind, so re-running the same fold in isolation
    vs. as part of the full 11-fold loop could silently produce a
    different result. This is exactly the kind of confound that made
    the earlier water-fix LOCO regression (0.3174 -> 0.2871) impossible
    to attribute to a real data effect vs. run-to-run noise -- don't
    repeat that here.

    Returns a dict with per-fold results: best_miou, best_epoch,
    per_class_f1, confusion_matrix, classes_present, history.
    """
    fold_seed = (LOCO_BASE_SEED + abs(hash(val_city)) % 100000) % (2**31 - 1)
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)
    if verbose:
        print(f'  [{val_city}] fold_seed={fold_seed} (deterministic per held-out city)')

    train_patches = [p for p in all_patches if p['city'] != val_city]
    val_patches = [p for p in all_patches if p['city'] == val_city]

    if not val_patches:
        if verbose:
            print(f'  [{val_city}] SKIPPED -- no patches for this city')
        return None

    train_cities = sorted(set(p['city'] for p in train_patches))
    if verbose:
        print(f'\n{"="*60}')
        print(f'LOCO FOLD: held-out = {val_city}')
        print(f'  Train: {len(train_patches)} patches from {len(train_cities)} cities')
        print(f'  Val:   {len(val_patches)} patches ({val_city})')

    from collections import Counter
    fold_counts = Counter(p['label'] for p in train_patches)
    counts_arr = np.array([fold_counts.get(c, 1) for c in CATEGORIES], dtype=np.float32)
    counts_arr = np.maximum(counts_arr, 1)
    fold_weights = 1.0 / counts_arr
    fold_weights = fold_weights / fold_weights.sum() * NUM_CLASSES
    fold_class_weights = torch.tensor(fold_weights, dtype=torch.float32).to(device)

    train_loader = DataLoader(GeoWatchDatasetResNet(train_patches, augment=True),
                               batch_size=batch_size, shuffle=True,
                               num_workers=0, pin_memory=True,
                               drop_last=True)
    val_loader = DataLoader(GeoWatchDatasetResNet(val_patches, augment=False),
                             batch_size=batch_size, shuffle=False,
                             num_workers=0, pin_memory=True)

    model = GeoWatchResNetSeg(num_classes=NUM_CLASSES, freeze_encoder=False).to(device)
    criterion = CombinedLoss(
        class_weights=fold_class_weights,
        ignore_index=IGNORE_INDEX,
        dice_weight=0.5,
        separation_weight=0.25,
        separation_margin=2.0,
        class_a_idx=CATEGORIES.index('paved_road'),
        class_b_idx=CATEGORIES.index('dense_informal_roofing'),
    )
    optimizer = optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': lr_encoder},
        {'params': model.decoder.parameters(), 'lr': lr_decoder},
    ], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_miou = 0.0
    best_epoch = 0
    best_state = None
    epochs_since_improve = 0
    history = {'train_loss': [], 'val_miou': []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        freeze_bn_stats(model.encoder)
        epoch_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss, _, _, _ = criterion(logits, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        history['train_loss'].append(avg_loss)

        model.eval()
        all_logits, all_masks = [], []
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                logits = model(images)
                all_logits.append(logits.cpu())
                all_masks.append(masks.cpu())
        all_logits_cat = torch.cat(all_logits, dim=0)
        all_masks_cat = torch.cat(all_masks, dim=0)
        val_miou = compute_miou(all_logits_cat, all_masks_cat, NUM_CLASSES, IGNORE_INDEX)
        history['val_miou'].append(val_miou)

        if val_miou > best_miou:
            best_miou = val_miou
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        if verbose:
            star = ' best' if epoch == best_epoch else ''
            print(f'  epoch {epoch:3d}/{num_epochs} | loss={avg_loss:.4f} | '
                  f'val_mIoU={val_miou:.4f}{star}')

        if epochs_since_improve >= patience:
            if verbose:
                print(f'  Early stop at epoch {epoch} (no improvement for {patience} epochs)')
            break

    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(masks.numpy())
    all_preds = np.concatenate([p.flatten() for p in all_preds])
    all_targets = np.concatenate([t.flatten() for t in all_targets])

    valid = all_targets != IGNORE_INDEX
    all_preds = all_preds[valid]
    all_targets = all_targets[valid]

    classes_present = sorted(set(all_targets.tolist()))
    class_names_present = [CATEGORIES[c] for c in classes_present]

    cm = confusion_matrix(all_targets, all_preds, labels=classes_present, normalize='true')
    f1_per_class = f1_score(all_targets, all_preds, labels=classes_present, average=None)

    if verbose:
        print(f'  FOLD RESULT: best_mIoU={best_miou:.4f} @ epoch {best_epoch} '
              f'| classes present: {class_names_present}')

    del model
    torch.cuda.empty_cache()

    return {
        'val_city': val_city,
        'best_miou': best_miou,
        'best_epoch': best_epoch,
        'train_cities': train_cities,
        'n_train': len(train_patches),
        'n_val': len(val_patches),
        'classes_present': class_names_present,
        'per_class_f1': dict(zip(class_names_present, f1_per_class.tolist())),
        'confusion_matrix': cm,
        'confusion_labels': class_names_present,
        'history': history,
    }


print('run_loco_fold() defined (6-band ResNet50 encoder, separation loss weight=0.25).')


Run LOCO across all 11 cities. Report the full per-fold table, never a bare mean.

In [ ]:
ALL_CITIES = sorted(set(p['city'] for p in all_patches))
print(f'Running LOCO across {len(ALL_CITIES)} cities: {ALL_CITIES}')
if len(ALL_CITIES) != 11:
    print(f'WARNING: expected 11 cities in all_patches, got {len(ALL_CITIES)}')
print('Early stopping (patience=8) means folds with fast convergence finish sooner.\n')

loco_results = {}
for city in ALL_CITIES:
    result = run_loco_fold(city, all_patches, num_epochs=30, patience=8, verbose=True)
    if result is not None:
        loco_results[city] = result

print(f'\n{"="*60}')
print(f'LOCO complete. {len(loco_results)}/{len(ALL_CITIES)} folds ran successfully.')

import numpy as np
fold_mious = [r['best_miou'] for r in loco_results.values()]
print(f'\nLOCO mean mIoU: {np.mean(fold_mious):.4f} (+/- {np.std(fold_mious):.4f}), '
      f'{len(fold_mious)} folds')
print('Per-fold table (report this in full, never just the mean):')
for city, r in sorted(loco_results.items(), key=lambda kv: -kv[1]["best_miou"]):
    print(f"  {city:<12} best_mIoU={r['best_miou']:.4f}  best_epoch={r['best_epoch']:>3}  "
          f"n_train={r['n_train']:>4}  n_val={r['n_val']:>3}  "
          f"classes_present={len(r['classes_present'])}/{NUM_CLASSES}")


## 9. PRODUCTION CELL

Trains the final production checkpoint on the FULL dataset (random monitoring split, NOT a
held-out city -- that number is NOT a generalization metric, only used to pick the best
epoch/checkpoint). **The real generalization number is the fresh LOCO mean from Section 8
above** -- fill it into the checkpoint dict below once that LOCO run finishes. Do not reuse
an older LOCO number from before this notebook's 6-band + water-data changes.


In [ ]:
import copy
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from collections import Counter


def freeze_bn_stats(module):
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.eval()


PROD_NUM_EPOCHS = 40
PROD_PATIENCE = 10
PROD_BATCH_SIZE = 16
PROD_LR_ENCODER = 1e-5
PROD_LR_DECODER = 3e-4
PROD_VAL_FRACTION = 0.10


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

shuffled = all_patches.copy()
random.shuffle(shuffled)
n_val = max(1, int(len(shuffled) * PROD_VAL_FRACTION))
monitor_val_patches = shuffled[:n_val]
train_patches = shuffled[n_val:]

print(f'Production training set: {len(train_patches)} patches')
print(f'Monitoring split (random, not held-out-city): {len(monitor_val_patches)} patches')
print(f'Cities represented in training: {sorted(set(p["city"] for p in train_patches))}')
print('NOTE: this monitoring mIoU is NOT a generalization metric. The real')
print('generalization number is the fresh LOCO result from Section 8 above.\n')

full_counts = Counter(p['label'] for p in train_patches)
counts_arr = np.array([full_counts.get(c, 1) for c in CATEGORIES], dtype=np.float32)
counts_arr = np.maximum(counts_arr, 1)
prod_weights = 1.0 / counts_arr
prod_weights = prod_weights / prod_weights.sum() * NUM_CLASSES
PROD_CLASS_WEIGHTS = torch.tensor(prod_weights, dtype=torch.float32).to(device)
print('Class weights (production, full dataset):')
for cat, w in zip(CATEGORIES, prod_weights):
    print(f'  {cat:28s} {w:.4f}')


train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(GeoWatchDatasetResNet(train_patches, augment=True),
                           batch_size=PROD_BATCH_SIZE, shuffle=True,
                           num_workers=0, pin_memory=True, drop_last=True,
                           generator=train_generator)
val_loader = DataLoader(GeoWatchDatasetResNet(monitor_val_patches, augment=False),
                         batch_size=PROD_BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

model = GeoWatchResNetSeg(num_classes=NUM_CLASSES, freeze_encoder=False).to(device)
criterion = CombinedLoss(
    class_weights=PROD_CLASS_WEIGHTS,
    ignore_index=IGNORE_INDEX,
    dice_weight=0.5,
    separation_weight=0.25,
    separation_margin=2.0,
    class_a_idx=CATEGORIES.index('paved_road'),
    class_b_idx=CATEGORIES.index('dense_informal_roofing'),
)
optimizer = optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': PROD_LR_ENCODER},
    {'params': model.decoder.parameters(), 'lr': PROD_LR_DECODER},
], weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=PROD_NUM_EPOCHS, eta_min=1e-6)

best_miou = 0.0
best_epoch = 0
best_state = None
epochs_since_improve = 0

print(f'\nTraining production model for up to {PROD_NUM_EPOCHS} epochs on {device}...')
print(f'Train: {len(train_loader)} batches | Monitor: {len(val_loader)} batches')
print('-' * 60)

for epoch in range(1, PROD_NUM_EPOCHS + 1):
    model.train()
    freeze_bn_stats(model.encoder)
    epoch_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss, _, _, _ = criterion(logits, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)

    model.eval()
    all_logits, all_masks = [], []
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            logits = model(images)
            all_logits.append(logits.cpu())
            all_masks.append(masks.cpu())
    all_logits_cat = torch.cat(all_logits, dim=0)
    all_masks_cat = torch.cat(all_masks, dim=0)
    monitor_miou = compute_miou(all_logits_cat, all_masks_cat, NUM_CLASSES, IGNORE_INDEX)

    if monitor_miou > best_miou:
        best_miou = monitor_miou
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1

    star = ' best' if epoch == best_epoch else ''
    print(f'Epoch {epoch:3d}/{PROD_NUM_EPOCHS} | loss={avg_loss:.4f} | '
          f'monitor_mIoU={monitor_miou:.4f}{star}')

    if epochs_since_improve >= PROD_PATIENCE:
        print(f'Early stop at epoch {epoch} (no improvement for {PROD_PATIENCE} epochs)')
        break

print(f'\nTraining complete. Best monitoring mIoU: {best_miou:.4f} at epoch {best_epoch}')

model.load_state_dict(best_state)

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/Geowatch/checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

fold_mious_for_checkpoint = [r['best_miou'] for r in loco_results.values()] if 'loco_results' in dir() else []
computed_loco_mean = float(np.mean(fold_mious_for_checkpoint)) if fold_mious_for_checkpoint else None
computed_loco_std = float(np.std(fold_mious_for_checkpoint)) if fold_mious_for_checkpoint else None

checkpoint = {
    'model_state_dict': model.state_dict(),
    'categories': CATEGORIES,
    'num_classes': NUM_CLASSES,
    'ignore_index': IGNORE_INDEX,
    'class_weights': PROD_CLASS_WEIGHTS.cpu(),
    'training_cities': sorted(set(p['city'] for p in all_patches)),
    'n_train_patches': len(train_patches),
    'n_monitor_patches': len(monitor_val_patches),
    'monitor_miou_at_save': best_miou,
    'monitor_epoch_at_save': best_epoch,
    'seed': SEED,
    'num_bands': 6,

    'loco_mean_miou': computed_loco_mean,
    'loco_std_miou': computed_loco_std,
    'loco_n_folds': len(fold_mious_for_checkpoint),

    'architecture': 'GeoWatchResNetSeg (ResNet50 SSL4EO-S12 MoCo dual-stem 6-band + DeepLabV3+, '
                     'paved_road/dense_informal_roofing separation loss)',
}

if computed_loco_mean is None:
    print('\nWARNING: loco_results not found in this session -- loco_mean_miou/std saved as '
          'None. Run Section 8 in this SAME session before saving, or manually fill these '
          'in from a separate LOCO run before trusting this checkpoint.')

local_path = '/content/geowatch_production_model.pth'
drive_path = f'{DRIVE_CHECKPOINT_DIR}/geowatch_production_model.pth'

torch.save(checkpoint, local_path)
import shutil
shutil.copy(local_path, drive_path)

print(f'\nSaved to Drive: {drive_path}')
print(f'Saved locally:  {local_path}')

from google.colab import files
print('\nTriggering download to your machine...')
files.download(local_path)


## 10. CAAT -- dynamic per-class confidence thresholds

In [ ]:
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

CHECKPOINT_PATH = '/content/geowatch_production_model.pth'

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

print(f'Loaded checkpoint: {CHECKPOINT_PATH}')
print(f'  Architecture: {checkpoint.get("architecture", "unknown")}')
print(f'  Training cities: {checkpoint.get("training_cities")}')
print(f'  Num bands: {checkpoint.get("num_bands", "unknown -- assume 6")}')
print(f'  Monitor mIoU at save: {checkpoint.get("monitor_miou_at_save")}')
print(f'  LOCO mean mIoU (real generalization number): '
      f'{checkpoint.get("loco_mean_miou")} +/- {checkpoint.get("loco_std_miou")} '
      f'({checkpoint.get("loco_n_folds")} folds)')

model = GeoWatchResNetSeg(num_classes=checkpoint['num_classes'], freeze_encoder=False).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

CATEGORIES = checkpoint['categories']
NUM_CLASSES = checkpoint['num_classes']
IGNORE_INDEX = checkpoint['ignore_index']

random.seed(42)
shuffled = all_patches.copy()
random.shuffle(shuffled)
n_val = max(1, int(len(shuffled) * 0.10))
monitor_val_patches = shuffled[:n_val]

print(f'\nRebuilt monitoring split: {len(monitor_val_patches)} patches '
      f'(should match n_monitor_patches={checkpoint.get("n_monitor_patches")} from checkpoint)')

if len(monitor_val_patches) != checkpoint.get('n_monitor_patches'):
    print('  WARNING: split size does not match the checkpoint -- all_patches may have '
          'changed since training. CAAT thresholds computed below may not be reliable.')

val_loader = DataLoader(GeoWatchDatasetResNet(monitor_val_patches, augment=False),
                         batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

all_logits, all_masks = [], []
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        logits = model(images)
        all_logits.append(logits.cpu())
        all_masks.append(masks.cpu())
check_miou = compute_miou(torch.cat(all_logits), torch.cat(all_masks), NUM_CLASSES, IGNORE_INDEX)

expected_miou = checkpoint.get('monitor_miou_at_save', None)
print(f'\nSanity check mIoU: {check_miou:.4f}  (checkpoint says: {expected_miou:.4f})')

if expected_miou is not None and abs(check_miou - expected_miou) > 0.01:
    print('  MISMATCH -- this does not look like the saved production model.')
    print('  DO NOT trust the CAAT thresholds below until this is resolved.')
else:
    print('  Matches checkpoint -- this IS the saved production model. Proceeding.')


def compute_caat_thresholds(model, val_loader, num_classes, device, ignore_index=255, percentile=10.0):
    """
    CAAT (Class-Adaptive Adaptive Thresholding). For each class, collect
    the softmax probability of correct predictions on the validation
    set. Set the threshold at the 10th percentile of those probabilities.
    """
    model.eval()
    class_correct_probs = {c: [] for c in range(num_classes)}

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)

            masks_device = masks.to(device)
            valid = masks_device != ignore_index

            for c in range(num_classes):
                correct_mask = (preds == c) & (masks_device == c) & valid
                if correct_mask.any():
                    correct_probs = probs[:, c][correct_mask].cpu().numpy()
                    class_correct_probs[c].extend(correct_probs.tolist())

    thresholds = {}
    print('\nCAAT thresholds (10th percentile of correct prediction confidence):')
    print('-' * 60)
    for c in range(num_classes):
        cat_name = CATEGORIES[c]
        if class_correct_probs[c]:
            thresh = float(np.percentile(class_correct_probs[c], percentile))
            thresh = max(thresh, 0.15)
        else:
            thresh = 0.50
            print(f'  {cat_name:<30} NO CORRECT PREDICTIONS -- threshold=0.50 (conservative)')
            thresholds[cat_name] = thresh
            continue

        n_samples = len(class_correct_probs[c])
        print(f'  {cat_name:<30} threshold={thresh:.3f}  (from {n_samples} correct predictions)')
        thresholds[cat_name] = thresh

    return thresholds


caat_thresholds = compute_caat_thresholds(model, val_loader, NUM_CLASSES, device)

caat_output = {
    'thresholds': caat_thresholds,
    'source_checkpoint': CHECKPOINT_PATH,
    'source_monitor_miou': checkpoint.get('monitor_miou_at_save'),
    'source_loco_mean_miou': checkpoint.get('loco_mean_miou'),
    'verified_sanity_check_miou': check_miou,
}

with open('/content/caat_thresholds.json', 'w') as f:
    json.dump(caat_output, f, indent=2)

print('\nSaved: /content/caat_thresholds.json')

import shutil
shutil.copy('/content/caat_thresholds.json',
            '/content/drive/MyDrive/Geowatch/checkpoints/caat_thresholds.json')
print('Also copied to Drive checkpoints folder, alongside the model.')


## 11. Dataset composition (reference)

In [ ]:
from collections import Counter
city_dist = Counter(p['city'] for p in all_patches)
print("Patches per city:")
for city, count in sorted(city_dist.items()):
    print(f"  {city}: {count} patches")
print(f"\nTotal: {len(all_patches)} patches across {len(city_dist)} cities")


In [ ]:
from collections import Counter

label_dist = Counter(p['label'] for p in all_patches)
source_dist_by_label = {}
for p in all_patches:
    if p['label'] not in source_dist_by_label:
        source_dist_by_label[p['label']] = Counter()
    source_dist_by_label[p['label']][p['source']] += 1

all_sources = sorted(set(p['source'] for p in all_patches))

col_width = 12
header = f"{'category':<30}" + ''.join(f"{s:>{col_width}}" for s in all_sources) + f"{'total':>10}"
print("Patches per class (by source):")
print(header)
print(chr(9472) * len(header))
for cat in CATEGORIES:
    sources = source_dist_by_label.get(cat, Counter())
    counts = [sources.get(s, 0) for s in all_sources]
    total = sum(counts)
    row = f"{cat:<30}" + ''.join(f"{c:>{col_width}}" for c in counts) + f"{total:>10}"
    print(row)

grand_total = sum(sum(source_dist_by_label.get(cat, Counter()).values()) for cat in CATEGORIES)
print(chr(9472) * len(header))
print(f"{'TOTAL (all categories)':<30}" + ' ' * (col_width * len(all_sources)) +
      f"{grand_total:>10}")
print(f"\n(should match len(all_patches) = {len(all_patches)} -- if not, some patch has "
      f"a label outside CATEGORIES, investigate)")
